In [1]:
import numpy as np
import torch
import torch.nn.functional as F
import deepxde as dde
import matplotlib.pyplot as plt
from scipy.constants import epsilon_0, e, electron_mass
from cherab.core.atomic import hydrogen
from cherab.openadas.parse.adf11 import parse_adf11

import sys 
import os

sys.path.append(os.path.abspath('../src/modules/'))

from sigmav2D import RateCoeff2D

Using backend: pytorch
Other supported backends: tensorflow.compat.v1, tensorflow, jax, paddle.
paddle supports more examples now and is recommended.


In [2]:
#Global constants
B = 1 #T
E_ION = 13.6 #eV
T_MAX = 7000 # eV
N_MAX = 2e13 # cm^-3
R = 72 # cm 
D_0 = 1e4 * (2 * np.sqrt(2 * np.pi * electron_mass) / 3) * (e / (4 * np.pi * epsilon_0 * B))**2 * (N_MAX * 1e6  / np.sqrt(T_MAX * e)) #cm^2 s^-1
KAPPA_0 = 4.7 * N_MAX * D_0 # cm^-1 s^-1
C = 2 * np.log(4 * np.pi * np.power(epsilon_0 * T_MAX * e, 1.5) / (e**3 * np.sqrt(N_MAX))) 
LAMBDA_N = 1.0
LAMBDA_T = 1.0
P_EXT = 1e18 #eV cm^-3 s^-1
VAR = 0.15
MU = 0.1 
N_0 = 1e8 # cm^{-3}

DTYPE = torch.float64
EPS   = 1e-40
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_default_device(DEVICE)

In [9]:
def ode_system(rho, y):
    sigma_ion = RateCoeff2D('../data/scd96_h.dat', hydrogen, n_max=N_MAX, T_max = T_MAX)
    sigma_rec = RateCoeff2D('../data/acd96_h.dat', hydrogen, n_max=N_MAX, T_max = T_MAX)
    p_rec = RateCoeff2D('../data/prb96_h.dat', hydrogen, n_max=N_MAX, T_max = T_MAX)
    p_rad_i = RateCoeff2D('../data/plt96_h.dat', hydrogen, n_max=N_MAX, T_max = T_MAX)
    p_rad_0 = RateCoeff2D('../data/prc96_h.dat', hydrogen, n_max=N_MAX, T_max = T_MAX)
    
    n = y[:, 0:1]
    T = y[:, 1:2]

    dn_rho = dde.grad.jacobian(y, rho, i=0)
    dT_rho = dde.grad.jacobian(y, rho, i=1)

    D_perp_eval = n / torch.sqrt(T) * ( torch.log(torch.pow(T + 1e-12, 3) / n) + C)
    func1 = rho * D_perp_eval * dn_rho
    func2 = rho * n * D_perp_eval * dT_rho
    
    df1_rho = dde.grad.jacobian(func1, rho)
    df2_rho = dde.grad.jacobian(func2, rho)

    pow_dep = ( (10 / 47) * (P_EXT * R * rho / (2 * VAR * np.pi * N_MAX * T_MAX * D_0)) 
                                              * torch.exp(-0.5 * (1 / VAR)**2 * (rho - MU)**2))
    
    coulomb_col = (15 * e**3 * (B * R * n)**2 * rho * (C + torch.log(T**3 / n)) 
                              / ( 47 * electron_mass * T_MAX * torch.sqrt(T)))

    power_den_rates = 10 * e * R**2 / (47 * N_MAX * T_MAX * D_0) * (p_rec.rate(n, T) + p_rad_i.rate(n, T) + (N_0 / N_MAX) * (p_rad_0.rate(n, T) ) ) 

    ode1 = df1_rho + (R**2 * rho / (D_0 * N_MAX) ) * ( sigma_ion.rate(n, T) - sigma_rec.rate(n, T))

    ode2 = df2_rho + (15/47) * (rho* dn_rho * dT_rho + T * df1_rho) + pow_dep - coulomb_col - power_den_rates

    return [ode1, ode2]

In [10]:
def boundary_l(x, on_boundary):
    return on_boundary and dde.utils.isclose(x[0], 0)

def boundary_r(x, on_boundary):
    return on_boundary and dde.utils.isclose(x[0], 1)

geom = dde.geometry.Interval(0.0, 1.0)

bcDirichlet_l_n = dde.icbc.DirichletBC(geom, lambda x: 1, boundary_l, component=0)
bcDirichlet_l_T = dde.icbc.DirichletBC(geom, lambda x: 1, boundary_l, component=1)

bcNeumann_l_n = dde.icbc.NeumannBC(geom, lambda x: 0, boundary_l, component=0)
bcNeumann_l_T = dde.icbc.NeumannBC(geom, lambda x: 0, boundary_l, component=1)

bcRobin_r_n = dde.icbc.RobinBC(geom, lambda X, y: -y / LAMBDA_N, boundary_r, component=0)
bcRobin_r_T = dde.icbc.RobinBC(geom, lambda X, y: -y / LAMBDA_T, boundary_r, component=1)

boundaries = [bcDirichlet_l_n, bcDirichlet_l_T, bcNeumann_l_n, bcNeumann_l_T, bcRobin_r_n, bcRobin_r_T]

data = dde.data.PDE(geom, ode_system, boundaries, 300, 2, num_test=300)

layer_size = [1] + [64] * 6 + [2]
activation = "tanh"
initializer = "Glorot normal"
net = dde.nn.FNN(layer_size, activation, initializer)
model = dde.Model(data, net)
model.compile(
    "adam", lr=0.001, loss_weights=[1]*8
)

losshistory, train_state = model.train(iterations=100)

Compiling model...
'compile' took 0.000279 s

Training model...

Step      Train loss                                                                          Test loss                                                                           Test metric
0         [nan, nan, 1.00e+00, 1.00e+00, 2.04e-02, 1.58e-01, 1.21e-01, 1.79e-01]              [2.17e+00, 1.87e+05, 1.00e+00, 1.00e+00, 2.04e-02, 1.58e-01, 1.21e-01, 1.79e-01]    []  

Best model at step 0:
  train loss: inf
  test loss: inf
  test metric: 

'train' took 0.068237 s

